## Packages

In [14]:
import json
import numpy as np


import tensorflow as tf
from tensorflow import keras
import nltk
nltk.download('punkt')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences3

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D


import pickle
import random

nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\91886\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\91886\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


ImportError: cannot import name 'pad_sequences3' from 'tensorflow.keras.preprocessing.sequence' (C:\Users\91886\anaconda3\lib\site-packages\keras\_tf_keras\keras\preprocessing\sequence\__init__.py)

## data loading

In [5]:
f = open('intents.json')
data = json.load(f)
data

{'intents': [{'tag': 'Cuts',
   'patterns': ['What to do if Cuts?',
    'how do u treat a cut',
    'How to cure Cuts?',
    'Which medicine to apply for Cuts?',
    'what to apply on cuts?',
    'Cuts',
    'cuts'],
   'responses': ['Wash the cut properly to prevent infection and stop the bleeding by applying pressure for 1-2minutes until bleeding stops. \n Apply Petroleum Jelly to make sure that the wound is moist for quick healing.\n Finally cover the cut with a sterile bandage. \n Pain relievers such as acetaminophen can be applied.'],
   'context_set': ''},
  {'tag': 'greeting',
   'patterns': ['Hi', 'How are you', 'Is anyone there?', 'Hello', 'Whats up'],
   'responses': ['Hi , i am your paramedic assistant'],
   'context_set': ''},
  {'tag': 'goodbye',
   'patterns': ['cya',
    'See you later',
    'Goodbye',
    'I am Leaving',
    'Have a Good day',
    'bye'],
   'responses': ['Sad to see you go :(', 'Talk to you later', 'Goodbye!'],
   'context_set': ''},
  {'tag': 'Abrasio

In [6]:
training_sentences = []
training_labels = []
labels = []
responses = []


for intent in data['intents']:
    for pattern in intent['patterns']:
        training_sentences.append(pattern)
        training_labels.append(intent['tag'])
    responses.append(intent['responses'])

    if intent['tag'] not in labels:
        labels.append(intent['tag'])

num_classes = len(labels)

In [7]:
training_sentences[:5]

['What to do if Cuts?',
 'how do u treat a cut',
 'How to cure Cuts?',
 'Which medicine to apply for Cuts?',
 'what to apply on cuts?']

In [8]:
training_labels[:10]

['Cuts',
 'Cuts',
 'Cuts',
 'Cuts',
 'Cuts',
 'Cuts',
 'Cuts',
 'greeting',
 'greeting',
 'greeting']

In [9]:
labels[:10]

['Cuts',
 'greeting',
 'goodbye',
 'Abrasions',
 'stings',
 'Splinter',
 'Sprains',
 'Strains',
 'Fever',
 'Nasal Congestion']

In [10]:
responses[:2]

[['Wash the cut properly to prevent infection and stop the bleeding by applying pressure for 1-2minutes until bleeding stops. \n Apply Petroleum Jelly to make sure that the wound is moist for quick healing.\n Finally cover the cut with a sterile bandage. \n Pain relievers such as acetaminophen can be applied.'],
 ['Hi , i am your paramedic assistant']]

## data preprocessing

In [11]:
lbl_encoder = LabelEncoder()
lbl_encoder.fit(training_labels)
training_labels = lbl_encoder.transform(training_labels)
training_labels

array([ 9,  9,  9,  9,  9,  9,  9, 41, 41, 41, 41, 41, 40, 40, 40, 40, 40,
       40,  1,  1,  1,  1,  1,  1, 45, 45, 45, 45, 45, 31, 31, 31, 31, 31,
       31, 32, 32, 32, 32, 32, 32, 32, 33, 33, 33, 33, 33, 33, 33, 33, 33,
       14, 14, 14, 14, 23, 23, 23, 23, 23, 23, 23,  8,  8,  8,  8,  8,  8,
        8, 30, 30, 30, 30, 30, 30, 17, 17, 17, 17, 17, 17, 29, 29, 29, 29,
       29, 29,  0,  0,  0,  0,  0,  3,  3,  3,  3,  3,  3,  2,  2,  2,  2,
        2,  2,  6,  6,  6,  6,  6,  6, 38, 38, 38, 38, 38, 38, 10, 10, 10,
       10, 10, 10, 16, 16, 16, 16, 16, 16, 20, 20, 20, 20, 20, 20, 21, 21,
       21, 21, 21, 22, 22, 22, 22, 22, 22, 42, 42, 42, 42, 42, 42, 26, 26,
       26, 26, 26, 26, 28, 28, 28, 28, 28, 28, 34, 34, 34, 34, 34, 34, 36,
       36, 36, 36, 36, 36, 37, 37, 37, 37, 37, 37, 24, 24, 24, 24, 24, 24,
       12, 12, 12, 12, 12,  5,  5,  5,  5,  5, 25, 25, 25, 25, 25, 25, 35,
       35, 35, 35, 35, 35, 35, 35, 43, 43, 43, 43, 43, 43, 18, 18, 18, 18,
       18, 18, 13, 13, 13

In [12]:
vocab_size = 1000
embedding_dim = 16
max_len = 20
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(training_sentences)
word_index = tokenizer.word_index
sequences = tokenizer.texts_to_sequences(training_sentences)
padded_sequences = pad_sequences(sequences, truncating='post', maxlen=max_len)
print(padded_sequences)

[[  0   0   0 ...   3   6  35]
 [  0   0   0 ...  11   4 104]
 [  0   0   0 ...   2  15  35]
 ...
 [  0   0   0 ...   8   4  53]
 [  0   0   0 ...  22   4  53]
 [  0   0   0 ...  15   4  53]]


## model building

In [13]:
model = Sequential()
model.add(Embedding(vocab_size, embedding_dim, input_length=max_len))
model.add(GlobalAveragePooling1D())
model.add(Dense(16, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy',optimizer='adam', metrics=['accuracy'])

model.summary()

C:\Users\91886\anaconda3\lib\site-packages\keras\src\layers\core\embedding.py:86: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ ?                           │     0 (unbuilt) │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## model training

In [12]:
epochs = 250
history = model.fit(padded_sequences, np.array(training_labels), epochs=epochs)

Epoch 1/250


9/9 [==============================] - 1s 6ms/step - loss: 3.8294 - accuracy: 0.0212
Epoch 2/250
9/9 [==============================] - 0s 3ms/step - loss: 3.8281 - accuracy: 0.0212
Epoch 3/250
9/9 [==============================] - 0s 2ms/step - loss: 3.8273 - accuracy: 0.0212
Epoch 4/250
9/9 [==============================] - 0s 2ms/step - loss: 3.8265 - accuracy: 0.0212
Epoch 5/250
9/9 [==============================] - 0s 4ms/step - loss: 3.8258 - accuracy: 0.0353
Epoch 6/250
9/9 [==============================] - 0s 2ms/step - loss: 3.8247 - accuracy: 0.0389
Epoch 7/250
9/9 [==============================] - 0s 3ms/step - loss: 3.8235 - accuracy: 0.0353
Epoch 8/250
9/9 [==============================] - 0s 2ms/step - loss: 3.8224 - accuracy: 0.0318
Epoch 9/250
9/9 [==============================] - 0s 3ms/step - loss: 3.8211 - accuracy: 0.0389
Epoch 10/250
9/9 [==============================] - 0s 4ms/step - loss: 3.8195 - accuracy: 0.0353
Epoch 11/250
9/9 [=========

9/9 [==============================] - 0s 3ms/step - loss: 1.6487 - accuracy: 0.7173
Epoch 81/250
9/9 [==============================] - 0s 4ms/step - loss: 1.6219 - accuracy: 0.7739
Epoch 82/250
9/9 [==============================] - 0s 3ms/step - loss: 1.5937 - accuracy: 0.7491
Epoch 83/250
9/9 [==============================] - 0s 2ms/step - loss: 1.5644 - accuracy: 0.7633
Epoch 84/250
9/9 [==============================] - 0s 3ms/step - loss: 1.5341 - accuracy: 0.7385
Epoch 85/250
9/9 [==============================] - 0s 4ms/step - loss: 1.5090 - accuracy: 0.7845
Epoch 86/250
9/9 [==============================] - 0s 4ms/step - loss: 1.4799 - accuracy: 0.8127
Epoch 87/250
9/9 [==============================] - 0s 2ms/step - loss: 1.4512 - accuracy: 0.8092
Epoch 88/250
9/9 [==============================] - 0s 3ms/step - loss: 1.4265 - accuracy: 0.8057
Epoch 89/250
9/9 [==============================] - 0s 4ms/step - loss: 1.4000 - accuracy: 0.8127
Epoch 90/250
9/9 [===============

9/9 [==============================] - 0s 2ms/step - loss: 0.2991 - accuracy: 0.9929
Epoch 164/250
9/9 [==============================] - 0s 2ms/step - loss: 0.2921 - accuracy: 0.9929
Epoch 165/250
9/9 [==============================] - 0s 2ms/step - loss: 0.2864 - accuracy: 0.9929
Epoch 166/250
9/9 [==============================] - 0s 4ms/step - loss: 0.2817 - accuracy: 0.9894
Epoch 167/250
9/9 [==============================] - 0s 2ms/step - loss: 0.2753 - accuracy: 0.9894
Epoch 168/250
9/9 [==============================] - 0s 3ms/step - loss: 0.2699 - accuracy: 0.9965
Epoch 169/250
9/9 [==============================] - 0s 2ms/step - loss: 0.2651 - accuracy: 0.9965
Epoch 170/250
9/9 [==============================] - 0s 3ms/step - loss: 0.2576 - accuracy: 0.9965
Epoch 171/250
9/9 [==============================] - 0s 2ms/step - loss: 0.2535 - accuracy: 0.9965
Epoch 172/250
9/9 [==============================] - 0s 2ms/step - loss: 0.2485 - accuracy: 0.9929
Epoch 173/250
9/9 [=====

## chatbot model

In [13]:
model.save("chat_model")

# to save the fitted tokenizer
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

# to save the fitted label encoder
with open('label_encoder.pickle', 'wb') as ecn_file:
    pickle.dump(lbl_encoder, ecn_file, protocol=pickle.HIGHEST_PROTOCOL)

INFO:tensorflow:Assets written to: chat_model\assets


INFO:tensorflow:Assets written to: chat_model\assets


In [14]:
import json 
import numpy as np
from tensorflow import keras
from sklearn.preprocessing import LabelEncoder


import random
import pickle

with open("intents.json") as file:
    data = json.load(file)


def chat():
    # load trained model
    model = keras.models.load_model('chat_model')

    # load tokenizer object
    with open('tokenizer.pickle', 'rb') as handle:
        tokenizer = pickle.load(handle)

    # load label encoder object
    with open('label_encoder.pickle', 'rb') as enc:
        lbl_encoder = pickle.load(enc)

    # parameters
    max_len = 20
    
    while True:
        print( "User: ", end="")
        inp = input()
        if inp.lower() == "quit":
            break

        result = model.predict(keras.preprocessing.sequence.pad_sequences(tokenizer.texts_to_sequences([inp]),
                                             truncating='post', maxlen=max_len))
        tag = lbl_encoder.inverse_transform([np.argmax(result)])

        for i in data['intents']:
            if i['tag'] == tag:
                print ("ChatBot:" , np.random.choice(i['responses']))
print( "Hii, Iam chatbot, I can help u with")
for i in labels:
  print(i, end = ",   ")
print("\n \n Start messaging with the bot (type quit to stop)! \n ")

chat()

Hii, Iam chatbot, I can help u with
Cuts,   greeting,   goodbye,   Abrasions,   stings,   Splinter,   Sprains,   Strains,   Fever,   Nasal Congestion,   Cough,   Sore Throat,   Gastrointestinal problems,   Skin problems,   Abdonominal Pain,   Bruises,   Broken Toe,   Choking,   Wound,   Diarrhea,   Frost bite,   Heat Exhaustion,   Heat Stroke,   Insect Bites,   nose bleed,   Pulled Muscle,   Rectal bleeding,   Sun Burn,   Testicle Pain,   Vertigo,   Normal Bleeding,   Eye Injury,   Chemical Burn,   Poison,   Teeth,   seizure,   Head Injury,   Fainting,   Headache,   Cold,   Rash,   snake bite,   animal bite,   Drowning,   CPR,   Fracture,   
 
 Start messaging with the bot (type quit to stop)! 
 



User: hi
1/1 [==============================] - 0s 67ms/step
ChatBot: Hi , i am your paramedic assistant
User: cute
1/1 [==============================] - 0s 16ms/step
ChatBot: 1)Encourage them to keep coughing to try to clear the blockage.
 2)Ask them to try to spit out the object if it's in their mouth.
 3)If coughing doesn't work, start back blows and Abdonominal thrusts
User: how to cure a cut
1/1 [==============================] - 0s 15ms/step
ChatBot: Wash the cut properly to prevent infection and stop the bleeding by applying pressure for 1-2minutes until bleeding stops. 
 Apply Petroleum Jelly to make sure that the wound is moist for quick healing.
 Finally cover the cut with a sterile bandage. 
 Pain relievers such as acetaminophen can be applied.
User: how do u cure a cute
1/1 [==============================] - 0s 16ms/step
ChatBot: Remove any stingers immediately.
 Some experts recommend scraping out the stinger with a credit card.
 Applying ice to the site may provide some 